# 12 — Hyperbolic quadrant: Encoding B data prep

**Encoding B** (dual-channel softmax):

- **Channel 1 (`B1`)**: user’s own rating → one-hot (same as Encoding A channel 1)
- **Channel 2 (`B2`)**: **population mean** rating for each movie (all MovieLens users), rounded to nearest half-star → one-hot

Channel 2 is filled **only** on user–movie pairs where the user has an observed rating (same sparsity mask as channel 1).

**Prerequisites:** notebook 09 (cohort + movie vocabulary `.npy` files).

In [ ]:
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd

root = Path.cwd().resolve()
if root.name == "notebooks":
    root = root.parent

rating_path = root / "data" / "rating.csv"
proc = root / "data" / "processed"
path_users = proc / "cohort_user_ids.npy"
path_movies = proc / "movie_vocab.npy"

for p in (rating_path, path_users, path_movies):
    assert p.exists(), f"Missing {p}"

RATING_LEVELS = np.array([0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0], dtype=np.float64)
K = len(RATING_LEVELS)
rating_to_idx = {float(r): i for i, r in enumerate(RATING_LEVELS)}

CHUNK_SIZE = 1_000_000
CSV_DTYPES = {"userId": "int32", "movieId": "int32", "rating": "float32"}

cohort_user_ids = np.load(path_users).astype(int).tolist()
movie_vocab = np.load(path_movies).astype(int).tolist()
cohort_set = set(cohort_user_ids)
vocab_set = set(movie_vocab)
user_to_row = {uid: i for i, uid in enumerate(cohort_user_ids)}
movie_to_col = {mid: j for j, mid in enumerate(movie_vocab)}
n_users, n_movies = len(cohort_user_ids), len(movie_vocab)

print(f"Cohort users: {n_users}")
print(f"Movie vocab:  {n_movies}")
print(f"User 666 in cohort: {666 in cohort_set}")

## 1. Population mean rating per movie (full dataset)

In [ ]:
def round_to_half_star(x):
    """Round to nearest 0.5 and clip to [0.5, 5.0]."""
    r = float(np.round(x * 2.0) / 2.0)
    return float(np.clip(r, 0.5, 5.0))


movie_sum = defaultdict(float)
movie_cnt = defaultdict(int)

print(f"Streaming {rating_path.name} for global movie means (vocab movies only) …")
for chunk in pd.read_csv(rating_path, dtype=CSV_DTYPES, chunksize=CHUNK_SIZE):
    sub = chunk[chunk["movieId"].isin(vocab_set)]
    if sub.empty:
        continue
    grp = sub.groupby("movieId")["rating"]
    for mid, s in grp.sum().items():
        movie_sum[int(mid)] += float(s)
    for mid, c in grp.count().items():
        movie_cnt[int(mid)] += int(c)

pop_mean_raw = np.zeros(n_movies, dtype=np.float64)
pop_mean_rounded = np.zeros(n_movies, dtype=np.float64)
pop_onehot_col = np.zeros((n_movies, K), dtype=np.float32)  # column template for ch2

for j, mid in enumerate(movie_vocab):
    if mid not in movie_cnt or movie_cnt[mid] == 0:
        raise ValueError(f"Movie {mid} in vocab has no ratings in full dataset")
    mu = movie_sum[mid] / movie_cnt[mid]
    mu_r = round_to_half_star(mu)
    pop_mean_raw[j] = mu
    pop_mean_rounded[j] = mu_r
    ki = rating_to_idx[mu_r]
    pop_onehot_col[j, ki] = 1.0

bin_counts = {float(b): int((pop_mean_rounded == b).sum()) for b in RATING_LEVELS}
print("\n=== Distribution of rounded population means (2000 movies) ===")
for b in RATING_LEVELS:
    print(f"  {b:.1f}: {bin_counts[float(b)]:4d}")

## 2. Build Encoding B tensors

In [ ]:
channelB1 = np.zeros((n_users, n_movies, K), dtype=np.float32)
channelB2 = np.zeros((n_users, n_movies, K), dtype=np.float32)

ch1_idx = np.full((n_users, n_movies), -1, dtype=np.int16)
ch2_idx = np.full((n_users, n_movies), -1, dtype=np.int16)

print("Streaming cohort ratings for channel B1 (user) + B2 mask …")
for chunk in pd.read_csv(rating_path, dtype=CSV_DTYPES, chunksize=CHUNK_SIZE):
    sub = chunk[chunk["userId"].isin(cohort_set) & chunk["movieId"].isin(vocab_set)]
    if sub.empty:
        continue
    for row in sub.itertuples(index=False):
        uid, mid, r = int(row.userId), int(row.movieId), float(row.rating)
        if r not in rating_to_idx:
            raise ValueError(f"Unexpected rating {r}")
        i, j = user_to_row[uid], movie_to_col[mid]
        k1 = rating_to_idx[r]
        k2 = int(pop_onehot_col[j].argmax())
        channelB1[i, j, k1] = 1.0
        channelB2[i, j, :] = pop_onehot_col[j]
        ch1_idx[i, j] = k1
        ch2_idx[i, j] = k2

n_obs = int((ch1_idx >= 0).sum())
print(f"Observed user-movie pairs: {n_obs:,}")

## 3. Save

In [ ]:
path_b1 = proc / "channelB1_softmax.npy"
path_b2 = proc / "channelB2_softmax.npy"
np.save(path_b1, channelB1)
np.save(path_b2, channelB2)
print(f"Saved {path_b1}  shape {channelB1.shape}")
print(f"Saved {path_b2}  shape {channelB2.shape}")

## 4. Verification

In [ ]:
obs_mask = ch1_idx >= 0
k1_obs = ch1_idx[obs_mask]
k2_obs = ch2_idx[obs_mask]

agree = int(np.sum(k1_obs == k2_obs))
strong_disagree = int(np.sum(np.abs(k1_obs - k2_obs) >= 3))
n_pairs = int(obs_mask.sum())

print("=== Agreement statistics (observed pairs only) ===")
print(f"Total observed pairs: {n_pairs:,}")
print(f"Channel index agreement (k1 == k2): {agree:,} ({100 * agree / n_pairs:.2f}%)")
print(f"Strong disagreement |k1-k2| >= 3: {strong_disagree:,} ({100 * strong_disagree / n_pairs:.2f}%)")

assert 666 in cohort_set
row_666 = user_to_row[666]
cols = np.where(ch1_idx[row_666] >= 0)[0]
assert len(cols) > 0
j = int(cols[0])
mid = movie_vocab[j]
k1 = int(ch1_idx[row_666, j])
k2 = int(ch2_idx[row_666, j])

print("\n=== User 666 example (one observed movie) ===")
print(f"movieId (vocab col {j}): {mid}")
print(f"user rating:              {RATING_LEVELS[k1]:.1f}")
print(f"raw population mean:      {pop_mean_raw[j]:.4f}")
print(f"rounded population mean:  {pop_mean_rounded[j]:.1f}")
print(f"channel 1 one-hot index:  {k1}")
print(f"channel 2 one-hot index:  {k2}")
print(f"channel 1 one-hot: {channelB1[row_666, j].astype(int).tolist()}")
print(f"channel 2 one-hot: {channelB2[row_666, j].astype(int).tolist()}")